In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay,
                             roc_auc_score, balanced_accuracy_score, recall_score)

# 1. Load data and hold out a test set
df = pd.read_csv("my_project_data.csv")                        # your file name

y = df["diagnosis"]                                            # your label column name
X = df.drop(columns=["diagnosis", "subject_id"])               # everything except the label and ID columns
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# 2. Model and baseline
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline = DummyClassifier(strategy="most_frequent")

# 3. Cross-validation on the training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["accuracy", "balanced_accuracy", "roc_auc"]
for name, clf in [("Baseline", baseline), ("Logistic regression", model)]:
    res = cross_validate(clf, X_train, y_train, cv=cv, scoring=metrics)
    print(name)
    for m in metrics:
        s = res[f"test_{m}"]
        print(f"  {m:18s} {s.mean():.3f} ± {s.std():.3f}")

# 4. Final fit and one-time test evaluation
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]   # probability of the positive class

print("\nTest set results")
print(classification_report(y_test, y_pred))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))
print("Sensitivity:", round(recall_score(y_test, y_pred, pos_label=1), 3))
print("Specificity:", round(recall_score(y_test, y_pred, pos_label=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

# 5. Visuals
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0])
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--")   # chance line
plt.tight_layout()
plt.show()